# Parameter finalization: SPI v2 teacher campaign

Managed Kaggle environment; source archive and cohort hashes are checked before
model loading. GPU1 runs Qwen3.5-9B in NF4 with thinking disabled. GPU0 runs
one YOLO-World detector independently of the text preflight; CLIP
image features are cached. Producer ownership and statuses are recorded.

The 12 text controls are synthetic rubric checks, not visual accuracy. Image
labels remain pseudo-labels, never human urgency truth. The 180-image cohort is
fully development-exposed. Incremental raw outputs and the recovery archive are
under /kaggle/working/results/spi; cell logs and executed notebook under
/kaggle/working/cell_evidence. External retrieval is required for durability.
If the four-class support gate passes, grouped CPU model selection follows the
labels. A separate VLM structural-control and natural-image diagnostic uses
actual scene predictions. No scheduler policy is trained.

In [ ]:
CELL_SOURCES=['import os\nos.environ[\'HF_HOME\']=\'/kaggle/temp/spi_models\'\nimport os,sys,json,base64,zipfile,io,hashlib\nfrom pathlib import Path\nprint("START immutable source setup",flush=True)\nROOT=Path(\'/kaggle/working/better_source\');ROOT.mkdir(exist_ok=False)\nwith zipfile.ZipFile(io.BytesIO(base64.b64decode(\'...\'))) as archive:\n    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in archive.namelist()):raise ValueError("unsafe archive")\n    archive.extractall(ROOT)\nSOURCE=json.loads((ROOT/\'SOURCE_MANIFEST.json\').read_text())\nfor name,expected in SOURCE[\'files\'].items():\n    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=expected:raise ValueError("source checksum mismatch")\nsys.path.insert(0,str(ROOT));os.environ[\'BETTER_SOURCE_MANIFEST\']=str(ROOT/\'SOURCE_MANIFEST.json\')\nfor key in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\'):os.environ[key]=\'1\'\nos.environ[\'YOLO_CONFIG_DIR\']=\'/tmp/better_yolo_config\'\nfrom BetterScheduler.cloud_notebook import CellRecorder\nrecorder=CellRecorder(CELL_SOURCES,Path(\'/kaggle/working/cell_evidence\'))\nwith recorder.cell(0):\n    print(\'Verified source\',SOURCE[\'commit\'],SOURCE[\'tag\'])\n    print(\'Actual interpreter\',sys.executable,sys.version)\n    print(\'Configuration:cached CLIP, GPU1 Qwen3.5-9B NF4, thinking OFF, CPU SPI head, server3600s\')', "with recorder.cell(1):\n    import subprocess,importlib.metadata as metadata,importlib.util\n    import torch\n    print('START SPI runtime setup; retain managed torch/CUDA',flush=True)\n    assert torch.cuda.device_count()==2,'Exactly two GPUs are required by the pinned Qwen9B loader'\n    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',\n        'transformers==5.3.0','huggingface-hub==1.5.0','accelerate==1.12.0',\n        'bitsandbytes==0.49.2','sentencepiece==0.2.2'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=300)\n    print(result.stdout,flush=True);result.check_returncode()\n    extra=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','ftfy'],\n        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=180)\n    print(extra.stdout,flush=True);extra.check_returncode()\n    extra=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps',\n        'ultralytics==8.3.237','git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6'],\n        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=300)\n    print(extra.stdout,flush=True);extra.check_returncode()\n    from transformers import Qwen3_5ForConditionalGeneration\n    assert not any(importlib.util.find_spec(x) is not None for x in ('fla','causal_conv1d'))\n    packages={x:metadata.version(x) for x in ('torch','transformers','huggingface-hub','accelerate','bitsandbytes','numpy','ultralytics','scikit-learn','Pillow','nbformat')}\n    from BetterScheduler.artifacts import write_json\n    write_json(Path('/kaggle/working/runtime.json'),{'python':sys.version,'interpreter':sys.executable,\n        'packages':packages,'devices':[torch.cuda.get_device_name(i) for i in (0,1)],'thinking':False})\n    os.environ['HF_HOME']='/kaggle/temp/spi_models'\n    print('END runtime',packages,flush=True)", "with recorder.cell(2):\n    import signal,time\n    started=time.monotonic()\n    def cutoff(signum,frame):raise TimeoutError('SPI whole-cell 3300-second budget reached')\n    signal.signal(signal.SIGALRM,cutoff);signal.alarm(3300)\n    from BetterScheduler.cloud_notebook import locate\n    from BetterScheduler.pilot_producers import snapshot_model\n    from BetterScheduler.qwen35_reviewer import Qwen35AnnotationReviewer,MODEL_ID,MODEL_REVISION\n    from BetterScheduler.model_settings import ReviewerSettings\n    from BetterScheduler.spi_final_campaign import execute\n    from BetterScheduler.spi_campaign import verify_inputs\n    INPUT=locate('input_manifest.json').parent\n    _,receipt=verify_inputs(INPUT)\n    print('Verified input BEFORE model download',receipt,flush=True)\n    print('START model download; pinned Qwen9B; no local laptop weights',flush=True)\n    model_path,revision=snapshot_model(MODEL_ID,MODEL_REVISION)\n    reviewer=Qwen35AnnotationReviewer(model_path,revision,\n        settings=ReviewerSettings(max_new_tokens=192,max_pixels=262144,image_max_side=512,prompt_variant='semantic_contract_v5'))\n    reviewer.maximum_generation_seconds=30;reviewer.maximum_input_tokens=4096;reviewer.require_finite_preflight=True\n    OUTPUT=execute(INPUT,Path('/kaggle/working/results/spi/20261027finalc'),reviewer,enable_scene=True,seconds=max(1,int(3200-(time.monotonic()-started))))\n    signal.alarm(0)\n    print('END actual SPI status',json.loads((OUTPUT/'status.json').read_text()),flush=True)", "with recorder.cell(3):\n    import zipfile\n    root=Path('/kaggle/working/results/spi')\n    files=[p for p in root.rglob('*') if p.is_file() and '/data/images/' not in str(p)]\n    if list(root.rglob('scene_interrupted.json')):files=[p for p in files if '/scene/' not in str(p)]\n    write_json(Path('/kaggle/working/spi_checksums.json'),{str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files})\n    with zipfile.ZipFile('/kaggle/working/spi_recovery.zip','w',zipfile.ZIP_DEFLATED) as archive:\n        for p in files:archive.write(p,str(p.relative_to(root)))\n    print('END saved models, raw labels, losses, statuses and cell logs; inspect experiment status',flush=True)"]
import os
os.environ['HF_HOME']='/kaggle/temp/spi_models'
import os,sys,json,base64,zipfile,io,hashlib
from pathlib import Path
print("START immutable source setup",flush=True)
ROOT=Path('/kaggle/working/better_source');ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    if any(not (ROOT/name).resolve().is_relative_to(ROOT) for name in archive.namelist()):raise ValueError("unsafe archive")
    archive.extractall(ROOT)
SOURCE=json.loads((ROOT/'SOURCE_MANIFEST.json').read_text())
for name,expected in SOURCE['files'].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest()!=expected:raise ValueError("source checksum mismatch")
sys.path.insert(0,str(ROOT));os.environ['BETTER_SOURCE_MANIFEST']=str(ROOT/'SOURCE_MANIFEST.json')
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[key]='1'
os.environ['YOLO_CONFIG_DIR']='/tmp/better_yolo_config'
from BetterScheduler.cloud_notebook import CellRecorder
recorder=CellRecorder(CELL_SOURCES,Path('/kaggle/working/cell_evidence'))
with recorder.cell(0):
    print('Verified source',SOURCE['commit'],SOURCE['tag'])
    print('Actual interpreter',sys.executable,sys.version)
    print('Configuration:cached CLIP, GPU1 Qwen3.5-9B NF4, thinking OFF, CPU SPI head, server3600s')

In [ ]:
with recorder.cell(1):
    import subprocess,importlib.metadata as metadata,importlib.util
    import torch
    print('START SPI runtime setup; retain managed torch/CUDA',flush=True)
    assert torch.cuda.device_count()==2,'Exactly two GPUs are required by the pinned Qwen9B loader'
    result=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
        'transformers==5.3.0','huggingface-hub==1.5.0','accelerate==1.12.0',
        'bitsandbytes==0.49.2','sentencepiece==0.2.2'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=300)
    print(result.stdout,flush=True);result.check_returncode()
    extra=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','ftfy'],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=180)
    print(extra.stdout,flush=True);extra.check_returncode()
    extra=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--no-deps',
        'ultralytics==8.3.237','git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6'],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=300)
    print(extra.stdout,flush=True);extra.check_returncode()
    from transformers import Qwen3_5ForConditionalGeneration
    assert not any(importlib.util.find_spec(x) is not None for x in ('fla','causal_conv1d'))
    packages={x:metadata.version(x) for x in ('torch','transformers','huggingface-hub','accelerate','bitsandbytes','numpy','ultralytics','scikit-learn','Pillow','nbformat')}
    from BetterScheduler.artifacts import write_json
    write_json(Path('/kaggle/working/runtime.json'),{'python':sys.version,'interpreter':sys.executable,
        'packages':packages,'devices':[torch.cuda.get_device_name(i) for i in (0,1)],'thinking':False})
    os.environ['HF_HOME']='/kaggle/temp/spi_models'
    print('END runtime',packages,flush=True)

In [ ]:
with recorder.cell(2):
    import signal,time
    started=time.monotonic()
    def cutoff(signum,frame):raise TimeoutError('SPI whole-cell 3300-second budget reached')
    signal.signal(signal.SIGALRM,cutoff);signal.alarm(3300)
    from BetterScheduler.cloud_notebook import locate
    from BetterScheduler.pilot_producers import snapshot_model
    from BetterScheduler.qwen35_reviewer import Qwen35AnnotationReviewer,MODEL_ID,MODEL_REVISION
    from BetterScheduler.model_settings import ReviewerSettings
    from BetterScheduler.spi_final_campaign import execute
    from BetterScheduler.spi_campaign import verify_inputs
    INPUT=locate('input_manifest.json').parent
    _,receipt=verify_inputs(INPUT)
    print('Verified input BEFORE model download',receipt,flush=True)
    print('START model download; pinned Qwen9B; no local laptop weights',flush=True)
    model_path,revision=snapshot_model(MODEL_ID,MODEL_REVISION)
    reviewer=Qwen35AnnotationReviewer(model_path,revision,
        settings=ReviewerSettings(max_new_tokens=192,max_pixels=262144,image_max_side=512,prompt_variant='semantic_contract_v5'))
    reviewer.maximum_generation_seconds=30;reviewer.maximum_input_tokens=4096;reviewer.require_finite_preflight=True
    OUTPUT=execute(INPUT,Path('/kaggle/working/results/spi/20261027finalc'),reviewer,enable_scene=True,seconds=max(1,int(3200-(time.monotonic()-started))))
    signal.alarm(0)
    print('END actual SPI status',json.loads((OUTPUT/'status.json').read_text()),flush=True)

In [ ]:
with recorder.cell(3):
    import zipfile
    root=Path('/kaggle/working/results/spi')
    files=[p for p in root.rglob('*') if p.is_file() and '/data/images/' not in str(p)]
    if list(root.rglob('scene_interrupted.json')):files=[p for p in files if '/scene/' not in str(p)]
    write_json(Path('/kaggle/working/spi_checksums.json'),{str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files})
    with zipfile.ZipFile('/kaggle/working/spi_recovery.zip','w',zipfile.ZIP_DEFLATED) as archive:
        for p in files:archive.write(p,str(p.relative_to(root)))
    print('END saved models, raw labels, losses, statuses and cell logs; inspect experiment status',flush=True)